# Project: a research programme with pandas

Merge the participants with the description of the projects, clean the data, summarize it and produce an Excel report.

The full explanations are in [programme_pandas.md](programme_pandas.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## Some data

Two fake tables: the participants (with duplicates and missing emails), and the projects.

In [ ]:
%%writefile generate_participants.py
"""Generate a fake list of participants of a research programme."""

import csv
import unicodedata

import numpy as np

rng = np.random.default_rng(seed=7)

first_names = ["Alice", "Bruno", "Chloé", "David", "Emma", "Farid", "Gaëlle",
               "Hugo", "Inès", "Julien", "Karima", "Louis", "Manon", "Nicolas",
               "Océane", "Pierre", "Quentin", "Rose", "Samir", "Théo"]
last_names = ["Martin", "Bernard", "Dubois", "Thomas", "Robert", "Richard",
              "Petit", "Durand", "Leroy", "Moreau", "Simon", "Laurent",
              "Lefebvre", "Michel", "Garcia", "David", "Bertrand", "Roux"]
institutes = ["INC", "INSB", "INP", "INSU", "INEE", "INSMI", "IN2P3", "INSIS"]
roles = ["PI", "Researcher", "Engineer", "PhD student", "Postdoc"]

rows = []
for i in range(1000):
    first = str(rng.choice(first_names))
    last = str(rng.choice(last_names))
    ascii_name = unicodedata.normalize("NFKD", f"{first}.{last}").encode("ascii", "ignore").decode()
    email = f"{ascii_name.lower()}{i}@example.fr"
    if rng.random() < 0.02:
        email = ""                                  # missing email
    rows.append({
        "first_name": first,
        "last_name": last,
        "email": email,
        "institute": str(rng.choice(institutes, p=[0.25, 0.2, 0.15, 0.1, 0.1, 0.05, 0.1, 0.05])),
        "role": str(rng.choice(roles, p=[0.1, 0.35, 0.2, 0.2, 0.15])),
        "project": f"P{rng.integers(1, 51):02d}",
    })

# a few people registered twice, with different capitalisation
for k in range(5):
    duplicate = dict(rows[k * 100])
    duplicate["email"] = duplicate["email"].upper()
    rows.append(duplicate)

with open("participants.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print("Written participants.csv with", len(rows), "rows")

In [ ]:
%%writefile generate_projects.py
"""Generate a fake table describing the 50 projects of the programme."""

import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=8)
projects = pd.DataFrame({
    "project": [f"P{i:02d}" for i in range(1, 51)],
    "theme": rng.choice(["Energy", "Health", "Environment", "Materials"], 50),
    "budget_keur": rng.integers(50, 400, 50) // 10 * 10,
    "start_year": rng.choice([2024, 2025, 2026], 50),
})
projects.to_csv("projects.csv", index=False)
print("Written projects.csv with", len(projects), "projects")

In [ ]:
%run generate_participants.py
%run generate_projects.py

## Step 1: loading and merging

`merge(..., on="project", how="left")` adds to each participant the
information of their project, like a VLOOKUP. `.str.strip()` and
`.str.lower()` apply to every value of a column. Beware:
`drop_duplicates("email")` would also merge all the people **without
email**, hence `remove_duplicates`.

In [ ]:
%%writefile programme_pd.py
"""Analyse the participants of a research programme with pandas."""

import pandas as pd


def load(participants_file, projects_file):
    """Read the two tables and merge them.

    Returns
    -------
    pandas.DataFrame
        One row per participant, with the information of their project.
    """
    participants = pd.read_csv(participants_file, keep_default_na=False)
    projects = pd.read_csv(projects_file)
    return participants.merge(projects, on="project", how="left")


def clean_emails(df):
    """Return a copy of the table with normalized emails (no spaces, lower case)."""
    df = df.copy()
    df["email"] = df["email"].str.strip().str.lower()
    return df


def duplicated_emails(df):
    """Return the rows whose email appears more than once (empty emails ignored)."""
    with_email = df[df["email"] != ""]
    return with_email[with_email.duplicated("email", keep=False)].sort_values("email")


def remove_duplicates(df):
    """Keep only the first registration of each email (empty emails are all kept)."""
    repeated = df.duplicated("email") & (df["email"] != "")
    return df[~repeated]


def summary_by_project(df):
    """Return one row per project: theme, budget, participants, budget per participant."""
    df = df.assign(is_pi=(df["role"] == "PI"))
    summary = df.groupby("project").agg(
        theme=("theme", "first"),
        budget_keur=("budget_keur", "first"),
        participants=("role", "size"),
        pis=("is_pi", "sum"),
    )
    summary["keur_per_participant"] = (summary["budget_keur"]
                                       / summary["participants"]).round(1)
    return summary.sort_values("participants", ascending=False)

In [ ]:
import pandas as pd
from programme_pd import load, clean_emails, summary_by_project

df = clean_emails(load("participants.csv", "projects.csv"))
print(df.shape)
df.head()

In [ ]:
summary_by_project(df).head()

In [ ]:
pd.crosstab(df["institute"], df["role"])

## Step 2: tests

A small hand-made table containing exactly the tricky cases.

In [ ]:
%%writefile test_programme_pd.py
"""Tests for programme_pd.py."""

import pandas as pd

from programme_pd import (clean_emails, duplicated_emails, remove_duplicates,
                          summary_by_project)


def make_table():
    return pd.DataFrame({
        "email": ["a@x.fr", " A@X.fr ", "b@x.fr", "", ""],
        "project": ["P01", "P02", "P01", "P03", "P03"],
        "role": ["PI", "Engineer", "Researcher", "PI", "Postdoc"],
        "theme": ["Energy", "Health", "Energy", "Materials", "Materials"],
        "budget_keur": [100, 50, 100, 30, 30],
    })


def test_clean_emails():
    df = clean_emails(make_table())
    assert df["email"].tolist() == ["a@x.fr", "a@x.fr", "b@x.fr", "", ""]


def test_clean_emails_does_not_modify_input():
    df = make_table()
    clean_emails(df)
    assert df.loc[1, "email"] == " A@X.fr "


def test_duplicated_emails_ignores_empty_emails():
    duplicates = duplicated_emails(clean_emails(make_table()))
    assert duplicates["email"].tolist() == ["a@x.fr", "a@x.fr"]


def test_remove_duplicates_keeps_empty_emails():
    df = remove_duplicates(clean_emails(make_table()))
    assert df["email"].tolist() == ["a@x.fr", "b@x.fr", "", ""]


def test_summary_by_project():
    summary = summary_by_project(make_table())
    assert summary.loc["P01", "participants"] == 2
    assert summary.loc["P01", "pis"] == 1
    assert summary.loc["P02", "pis"] == 0
    assert summary.loc["P01", "keur_per_participant"] == 50.0

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 3: the Excel report

In [ ]:
%%writefile report_pd.py
"""Produce an Excel report on the participants of the programme."""

import argparse

import matplotlib.pyplot as plt
import pandas as pd

from programme_pd import (clean_emails, duplicated_emails, load,
                          remove_duplicates, summary_by_project)


def main():
    parser = argparse.ArgumentParser(description="Programme report with pandas.")
    parser.add_argument("participants", help="CSV file of participants")
    parser.add_argument("projects", help="CSV file of projects")
    parser.add_argument("--output", default="report.xlsx", help="Excel file")
    args = parser.parse_args()

    df = clean_emails(load(args.participants, args.projects))
    duplicates = duplicated_emails(df)
    missing = df[df["email"] == ""]
    df = remove_duplicates(df)
    summary = summary_by_project(df)
    roles = pd.crosstab(df["institute"], df["role"], margins=True)

    print(f"{len(df)} participants in {df['project'].nunique()} projects")
    print(f"{len(duplicates) // 2} duplicated registrations removed")
    print(f"{len(missing)} participants without email")
    print("Projects without PI:", ", ".join(summary[summary["pis"] == 0].index))

    with pd.ExcelWriter(args.output) as writer:
        summary.to_excel(writer, sheet_name="Projects")
        roles.to_excel(writer, sheet_name="Institutes x roles")
        duplicates.to_excel(writer, sheet_name="Duplicates", index=False)
        missing.to_excel(writer, sheet_name="Missing emails", index=False)
    print("Report written in", args.output)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    df["theme"].value_counts().plot.bar(ax=axes[0], rot=0)
    axes[0].set_ylabel("Participants")
    summary.plot.scatter(x="participants", y="budget_keur", ax=axes[1])
    axes[1].set_ylabel("Budget (k€)")
    fig.tight_layout()
    fig.savefig("report.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run report_pd.py participants.csv projects.csv

In [ ]:
# Read the report back: one DataFrame per sheet
sheets = pd.read_excel("report.xlsx", sheet_name=None)
{name: table.shape for name, table in sheets.items()}

## Going further

**1.** Build a table with the number of participants for each institute (rows) and theme (columns).

<details><summary>Answer</summary>

```python
df.pivot_table(index="institute", columns="theme", values="email", aggfunc="count")
```

</details>

In [ ]:
# Your code here

**2.** For each project, build the list of emails separated by `; `, and save it as a CSV file.

<details><summary>Answer</summary>

```python
with_email = df[df["email"] != ""]
mailing = with_email.groupby("project")["email"].agg("; ".join)
mailing.to_csv("mailing.csv")
```

</details>

In [ ]:
# Your code here

**3.** What is the total budget of each theme? Careful: in the merged table, the budget appears once per participant.

<details><summary>Answer</summary>

```python
df.drop_duplicates("project").groupby("theme")["budget_keur"].sum()
```

</details>

In [ ]:
# Your code here